In [3]:
from pyomo.environ import *
import numpy as np

# -------------------------------------------------
# Load QUBO matrix
# -------------------------------------------------
Q = np.loadtxt("Data/QUBO_Matrix_1.txt", dtype=float)
n = Q.shape[0]
print(n)
# -------------------------------------------------
# Create Pyomo model
# -------------------------------------------------
model = ConcreteModel(name="QUBO_Model")

model.I = RangeSet(0, n-1)
model.x = Var(model.I, domain=Binary)

# -------------------------------------------------
# QUBO Objective: x^T Q x
# -------------------------------------------------
def qubo_objective(model):
    return sum(
        Q[i, j] * model.x[i] * model.x[j]
        for i in range(n)
        for j in range(n)
    )

model.obj = Objective(rule=qubo_objective, sense=minimize)

# Solve
# To use scip replace cplex_direct with scip
solver = SolverFactory('scip', solver_io='nl')
result = solver.solve(model, tee=False)



K = 5   # keep 1 if you want pure minimization

for i in range(K):

    solver.solve(model, tee=False)

    print(f"iteration {i}")
    model.display()


8
CPLEX 22.1.1.0: Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
iteration 0
Model QUBO_Model

  Variables:
    x : Size=8, Index=I
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          0 :     0 :   1.0 :     1 : False : False : Binary
          1 :     0 :   1.0 :     1 : False : False : Binary
          2 :     0 :   1.0 :     1 : False : False : Binary
          3 :     0 :   0.0 :     1 : False : False : Binary
          4 :     0 :   0.0 :     1 : False : False : Binary
          5 :     0 :   1.0 :     1 : False : False : Binary
          6 :     0 :   1.0 :     1 : False : False : Binary
          7 :     0 :   0.0 :     1 : False : False : Binary

  Objectives:
    obj : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : -6889.0

  Constraints:
    None
iteration 1
Model QUBO_Model

  Variables:
    x : Size=8, Index=I
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          0 :     0 :   1.0 :     1 : Fa

# To fetch top 10 solutions

In [1]:
from pyomo.environ import *
import numpy as np
import os

# -------------------------------------------------
# Load QUBO matrix
# -------------------------------------------------
qubo_file = "Data/QUBO_Matrix_1.txt"
Q = np.loadtxt(qubo_file)
n = Q.shape[0]

print("Matrix size:", n)

base_name = os.path.splitext(os.path.basename(qubo_file))[0]

K = 5
solvers = ["scip", "cplex"]

def build_model():
    model = ConcreteModel("QUBO_TopK")
    model.I = RangeSet(0, n - 1)
    model.x = Var(model.I, domain=Binary)

    model.obj = Objective(
        expr=sum(Q[i, j] * model.x[i] * model.x[j]
                 for i in range(n) for j in range(n)),
        sense=minimize
    )

    model.exclude = ConstraintList()
    return model

for solver_name in solvers:

    solver = SolverFactory(solver_name)

    if not solver.available():
        print(f"{solver_name.upper()} not available. Skipping.")
        continue

    print(f"\n==============================")
    print(f"Solving with {solver_name.upper()}")
    print(f"==============================")

    model = build_model()
    solutions = []

    for k in range(K):
        result = solver.solve(model, tee=True)

        if result.solver.termination_condition != TerminationCondition.optimal:
            print("No more optimal solutions.")
            break

        bitstring = [int(value(model.x[i])) for i in range(n)]
        obj_val = value(model.obj)

        solutions.append((obj_val, bitstring))

        # 🔹 MODIFIED NOTEBOOK OUTPUT
        print(f"\nSolution {k+1}")
        print("Objective:", obj_val)
        print("Bitstring:", bitstring)

        model.exclude.add(
            sum(
                (1 - model.x[i]) if bitstring[i] == 1 else model.x[i]
                for i in range(n)
            ) >= 1
        )

    output_file = f"{base_name}_{solver_name}.txt"

    with open(output_file, "w") as f:
        f.write(f"QUBO File      : {qubo_file}\n")
        f.write(f"Solver         : {solver_name.upper()}\n")
        f.write(f"Matrix size    : {n} x {n}\n")
        f.write(f"Top-K          : {len(solutions)}\n\n")

        for idx, (obj, bits) in enumerate(solutions, start=1):
            f.write(f"Solution {idx}\n")
            f.write(f"Objective: {obj}\n")
            f.write(f"Bitstring: {bits}\n\n")

    print(f"Results saved to: {output_file}")


Matrix size: 8

Solving with SCIP
SCIP version 9.2.4 [precision: 8 byte] [memory: block] [mode: optimized] [LP solver: SoPlex 7.1.6] [GitHash: a94f778ac8]
Copyright (c) 2002-2025 Zuse Institute Berlin (ZIB)

External libraries: 
  SoPlex 7.1.6         Linear programming solver developed at Zuse Institute Berlin (soplex.zib.de) [GitHash: fe6bd5d8]
  CppAD 20180000.0     Algorithmic Differentiation of C++ algorithms developed by B. Bell (github.com/coin-or/CppAD)
  ZLIB 1.3.1           General purpose compression library by J. Gailly and M. Adler (zlib.net)
  GMP 6.3.0            GNU Multiple Precision Arithmetic Library developed by T. Granlund (gmplib.org)
  ZIMPL 3.6.2          Zuse Institute Mathematical Programming Language developed by T. Koch (zimpl.zib.de)
  AMPL/MP 690e9e7      AMPL .nl file reader library (github.com/ampl/mp)
  PaPILO 2.4.4         parallel presolve for integer and linear optimization (github.com/scipopt/papilo) (built with TBB) [GitHash: 7c826d50]
  Nauty 2.8.